# Phase 6: BERT Teacher Model Fine-Tuning

This notebook walks through fine-tuning a full-precision `bert-base-uncased` sequence classifier to serve as our high-precision local teacher/oracle (Tier 3) in the BrainDump.AI ensemble.

### Pipeline Steps:
1. **Load Labeled Data**: Extract text and tags from `seed_data.json`.
2. **Preprocess & Tokenize**: Encode thoughts using the BERT Tokenizer.
3. **Fine-Tuning**: Run Hugging Face `Trainer` on CPU inside Docker.
4. **Export Model**: Save model weights to `models/teacher_bert/`.
5. **Interactive Testing**: Validate predictions visually with Matplotlib.

In [ ]:
import sys
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Resolve ROOT dynamically by looking for requirements.txt in current or parent directory
cwd = Path(".").resolve()
if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    ROOT = Path("..").resolve()

sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "training" / "data" / "seed_data.json"
MODEL_PATH = ROOT / "models" / "teacher_bert"
CATEGORIES = ["PERSONAL", "FINANCIAL", "PROJECTS", "ADMIN", "AUTOMATION"]

### Step 1: Wrap Labeled Data in a PyTorch Dataset

In [ ]:
class ThoughtsDataset(torch.utils.data.Dataset):
    """Wrapper to feed tokenized features directly to BERT model inputs."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

### Step 2: Load and Visualize Labeled Seed Data

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [item["text"] for item in data]
labels = [item["label"] for item in data]

print(f"Loaded {len(texts)} sample thoughts.")

# Plot the sample counts distribution using Matplotlib
df = pd.DataFrame({"label": labels})
counts = df['label'].value_counts()
colors = ['#3b82f6', '#22c55e', '#06b6d4', '#a855f7', '#f59e0b']

plt.figure(figsize=(7, 4.5))
bars = plt.bar(counts.index, counts.values, color=colors[:len(counts)], alpha=0.8, edgecolor='black')
plt.title('Category Sample Counts (Teacher Dataset)', fontsize=11, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.2, str(yval), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Step 3: Tokenization & Split

In [ ]:
# Map category tags to indices
label_map = {cat: i for i, cat in enumerate(CATEGORIES)}
label_ids = [label_map[l] for l in labels]

# Load pre-trained BERT Tokenizer
print("Loading tokenizer...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Split into 80/20 train/test set
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, label_ids, test_size=0.2, random_state=42, stratify=label_ids
)

print("Encoding text fields...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=64)

train_dataset = ThoughtsDataset(train_encodings, train_labels)
val_dataset = ThoughtsDataset(val_encodings, val_labels)
print(f"Inputs generated. Shape of Train: {len(train_dataset)}, Validation: {len(val_dataset)}")

### Step 4: Download & Load Pre-trained BERT Model

In [ ]:
print("Loading Sequence Classification Head on bert-base-uncased...")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5)
print(f"BERT Loaded successfully! Total params: {sum(p.numel() for p in model.parameters()):,} parameters.")

### Step 5: Fine-Tune BERT model using Hugging Face Trainer

*Note: We explicitly enforce `no_cuda=True` to compile training safely on CPU.*

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,                # 3 epochs is enough for seed convergence
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=5,
    evaluation_strategy="epoch",
    save_strategy="no",
    no_cuda=True,                      # force CPU inside docker
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Training beginning...")
trainer.train()

### Step 6: Export Teacher Model Assets

In [ ]:
print(f"Saving trained model files to {MODEL_PATH}...")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(str(MODEL_PATH))
model.save_pretrained(str(MODEL_PATH))
print("BERT Teacher Saved! Ready for active routing on reload.")

### Step 7: Local Interactive Prediction Sandbox

In [ ]:
test_phrases = [
    "buy dental floss and toothpaste",
    "calculate budget projection for the next 6 months",
    "implement stripe webhooks in python",
    "meeting to align with stakeholders at 2pm",
    "schedule cron backup script for databases",
    "need to fix my broken tires on the car"
]

model.eval()
results = []

for phrase in test_phrases:
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True, max_length=64)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].numpy()
    
    best_idx = np.argmax(probs)
    results.append({
        "phrase": phrase,
        "category": CATEGORIES[best_idx],
        "confidence": probs[best_idx]
    })

# Visualize predicted class confidence outputs
fig, ax = plt.subplots(figsize=(9, 4))
y_pos = np.arange(len(results))
confidences = [r["confidence"] for r in results]
labels_with_cat = [f"[{r['category']}] {r['phrase']}" for r in results]

bars = ax.barh(y_pos, confidences, color='#39ff14', alpha=0.8, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_with_cat, fontsize=9.5, fontweight='bold')
ax.invert_yaxis()  # top-down order
ax.set_xlabel('Prediction Confidence')
ax.set_title('BERT Teacher Prediction Sandbox (Local Smoke Test)')
ax.set_xlim(0, 1.15)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.1%}', 
            ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()